## R24 U01 Survey Links
#### [Renumeration](https://ufl.yul1.qualtrics.com/responses/#/surveys/SV_56kKZWusksy6LCC)
#### [R24_Post](https://ufl.yul1.qualtrics.com/responses/#/surveys/SV_1B4qIsmRrAJirl4)
#### [U01_Post](https://ufl.yul1.qualtrics.com/responses/#/surveys/SV_88tbPYLLLIHQDBk)
#### [Prescreener](https://ufl.yul1.qualtrics.com/responses/#/surveys/SV_7ZIq0HtZGmIJqia)


In [1]:
import pandas as pd
import os
import numpy as np
import pyodbc
import csv

DATA_DIRECTORY = "Data"
OUTPUT_DIRECTORY = "Outputs"
SUBFILE_DIRECTORY_NAME = "Subfiles"
OLD_R24_LOGS = "R24_Logs_Old.csv"
OLD_R24_LOGS_PATH = os.path.join(DATA_DIRECTORY, OLD_R24_LOGS)

print("Please choose a folder in this directory to read data from:")
print("📁 Folder Options:")
available_folders = [item for item in os.listdir(DATA_DIRECTORY) if os.path.isdir(os.path.join(DATA_DIRECTORY, item))]
print(available_folders)

# User input (e.g., "09-15")
directory = input("📣 Your selection: ").strip()
selected_dir = os.path.join(DATA_DIRECTORY, directory)
# Create the directory if it doesn't exist
if not os.path.exists(selected_dir):
    os.makedirs(selected_dir, exist_ok=True)
    print(f"📂 Created new input folder: {selected_dir}")
else:
    print(f"✅ Using existing input folder: {selected_dir}")

# Create new directory in Outputs
SELECTED_OUTPUT_DIRECTORY = os.path.join(OUTPUT_DIRECTORY, directory)
os.makedirs(SELECTED_OUTPUT_DIRECTORY, exist_ok=True)
print(f"📂 Output folder created at: {SELECTED_OUTPUT_DIRECTORY}")

# Create output directory with the same name under OUTPUT_DIRECTORY
SUBFILE_DIRECTORY = os.path.join(SELECTED_OUTPUT_DIRECTORY, "Subfiles")
os.makedirs(SUBFILE_DIRECTORY, exist_ok=True)
print(f"📂 Subfiles folder created at: {SUBFILE_DIRECTORY}")


PRESCREENER = "Prescreener.csv"
U01_LOGS = "U01_Logs.csv"
U01_POST = "U01_Post.csv"
R24_LOGS = "R24_Logs.csv"
R24_POST = "R24_Post.csv"
RENUMERATION = "Remuneration.csv"

PRESCREENER_PATH = os.path.join(selected_dir, PRESCREENER)
U01_LOGS_PATH = os.path.join(selected_dir, U01_LOGS)
U01_POST_PATH = os.path.join(selected_dir, U01_POST)
R24_LOGS_PATH = os.path.join(selected_dir, R24_LOGS)
R24_POST_PATH = os.path.join(selected_dir, R24_POST)
RENUMERATION_PATH = os.path.join(selected_dir, RENUMERATION)

# MUST_KEEP_IDS = ['R_5ngxcAdXeSIG2wp', 'R_2zd2dwY8SpNUdsE', 'R_6XuCGxChR0V174R', 'R_3YrhGZr8W8BTbDD', 'R_3uhrxqMMSojC0kV', 'R_75AAn44RwQwhk4J', 'R_572u6KRWn4n6BVQ', 'R_6EANjXVNcmXKRmw', 'R_5eUL0r35Nn5miEp', 'R_5NmpOuPqyLsiMX3']
MUST_KEEP_IDS = []

Please choose a folder in this directory to read data from:
📁 Folder Options:
['February 13 2026', 'February 2 2026', 'February 9 2026', 'February 23 2026', 'February 6 2026', 'February 20 2026', 'February 16 2026', 'February 27 2026', 'March 2 2026']


📣 Your selection:  March 2 2026


✅ Using existing input folder: Data/March 2 2026
📂 Output folder created at: Outputs/March 2 2026
📂 Subfiles folder created at: Outputs/March 2 2026/Subfiles


In [ ]:
# (0) FETCH SQL LOGS FROM VERG DB
import json

conn_str = (
    "GET FROM DEV TEAM"
)

# Open connection
conn = pyodbc.connect(conn_str)

R24_QUERY_STRING = r"""
SELECT *
FROM R24U01
WHERE ID LIKE 'R\_%' ESCAPE '\'
  AND Version = 'r24';

"""

U01_QUERY_STRING = r"""
SELECT *
FROM R24U01
WHERE ID LIKE 'R\_%' ESCAPE '\'
  AND Version = 'u01';
"""

def count_results(numres):
    total = 0
    if isinstance(numres, list):
        for entry in numres:
            if isinstance(entry, dict) and "results" in entry:
                if isinstance(entry["results"], list):
                    total += len(entry["results"])
    return total

def process_studyresults(val):
    """Convert NumberOfResults arrays into counts."""
    if not isinstance(val, str):
        return val
    try:
        data = json.loads(val)
        if isinstance(data, str):  # double-encoded JSON
            data = json.loads(data)
    except Exception:
        return val

    def walk(x):
        if isinstance(x, dict):
            if "NumberOfResults" in x and isinstance(x["NumberOfResults"], list):
                x["NumberOfResults"] = count_results(x["NumberOfResults"])
            for k, v in x.items():
                x[k] = walk(v)
        elif isinstance(x, list):
            return [walk(i) for i in x]
        return x

    return json.dumps(walk(data), ensure_ascii=False)

def clean_and_save(df, filepath, json_cols=None):
    if json_cols:
        for col in json_cols:
            if col in df.columns:
                df[col] = df[col].apply(process_studyresults)
    df.to_csv(filepath, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8-sig")
    print(f"✅ Saved {filepath} ({len(df)} rows)")


DOWNLOAD_R24_DF = pd.read_sql(R24_QUERY_STRING, conn)
DOWNLOAD_U01_DF = pd.read_sql(U01_QUERY_STRING, conn)

DOWNLOAD_R24_PATH = os.path.join(selected_dir, R24_LOGS)
DOWNLOAD_U01_PATH = os.path.join(selected_dir, U01_LOGS)


clean_and_save(DOWNLOAD_R24_DF, DOWNLOAD_R24_PATH, json_cols=["StudyResults"])
print(f"✅ R24 Logs saved to {DOWNLOAD_R24_PATH}")

clean_and_save(DOWNLOAD_U01_DF, DOWNLOAD_U01_PATH, json_cols=["StudyResults"])
print(f"✅ U01 Logs saved to {DOWNLOAD_U01_DF}")

/var/folders/pg/3jpkvk2j35b3prbshr_8q1s00000gn/T/ipykernel_24793/3382792974.py:72: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  DOWNLOAD_R24_DF = pd.read_sql(R24_QUERY_STRING, conn)
/var/folders/pg/3jpkvk2j35b3prbshr_8q1s00000gn/T/ipykernel_24793/3382792974.py:73: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  DOWNLOAD_U01_DF = pd.read_sql(U01_QUERY_STRING, conn)


✅ Saved Data/March 2 2026/R24_Logs.csv (1451 rows)
✅ R24 Logs saved to Data/March 2 2026/R24_Logs.csv
✅ Saved Data/March 2 2026/U01_Logs.csv (3873 rows)
✅ U01 Logs saved to                      ID            DateTime Version  VisitNum  \
0     R_3WDtubQKBOgWwWg 2025-06-16 09:35:41     u01         1   
1     R_80w682RPKaDlLRl 2025-10-08 14:26:41     u01         1   
2     R_5KMm6CyMtZ1tjix 2025-10-08 13:51:40     u01         1   
3     R_85m1HGg9lq6pjOh 2025-10-08 08:21:29     u01         1   
4     R_572u6KRWn4n6BVQ 2025-10-08 11:16:45     u01         2   
...                 ...                 ...     ...       ...   
3868  R_6Db8rGf2QYQ2Khb 2025-12-16 11:55:27     u01         1   
3869  R_7s4AoAYDrExASDg 2025-12-15 08:40:13     u01         1   
3870  R_5Fmel0kYam9O9nr 2025-12-15 08:41:15     u01         4   
3871  R_1mJlg0dZ9AOSyqe 2025-12-15 09:16:37     u01         1   
3872  R_3qDL3emRTLTBX1j 2026-01-07 18:37:34     u01         2   

     InterventionType                         

In [3]:
# (1) PRESCREENER ------------------------------------------------------------------------------------

print("\n ✦━━━━━━━━✧━━━━━━━━✦❘༻ CLEANING PRESCREENER ༺❘✦━━━━━━━━✧━━━━━━━━✦ \n")

df_prescreener = pd.read_csv(PRESCREENER_PATH, skiprows=[1, 2])

print("ⓘ Dropping unused columns & rows...")

# Drop the specified columns
columns_to_drop = [
    'RecipientLastName', 'RecipientFirstName', 'RecipientEmail',
    'ExternalReference',
    'DistributionChannel',
    'Q_AmbiguousTextPresent', 'Q_AmbiguousTextQuestions',
    'Q_StraightliningCount', 'Q_StraightliningPercentage',
    'Q_StraightliningQuestions', 'Q_UnansweredPercentage',
    'Q_UnansweredQuestions'
]

df_prescreener = df_prescreener.drop(columns=columns_to_drop)
print("DF Length: ", len(df_prescreener))

df_prescreener = df_prescreener.rename(columns={'ResponseId': 'ID'})

df_prescreener = df_prescreener[
    (
        (df_prescreener["Q_RecaptchaScore"] >= 0.5) &
        (df_prescreener["Q_RelevantIDDuplicateScore"] < 75) &
        (df_prescreener["Q_RelevantIDFraudScore"] < 30)
    )
    | df_prescreener["ID"].isin(MUST_KEEP_IDS)
]

print("DF Length after Bots: ", len(df_prescreener))

# (Optional) Save the result
df_prescreener.to_csv(os.path.join(SUBFILE_DIRECTORY, "Prescreener_cleaned.csv"), index=False)

print("\n ✔ DONE! Check for Prescreener_cleaned.csv")

# Make the new unified column
df_prescreener["Email_Address"] = df_prescreener[["U01_Email_18-49", "U01_Email_50+", "R24_Email"]].bfill(axis=1).iloc[:, 0]
print("DF Length after Email Unification: ", len(df_prescreener))

# Check for rows where more than one email column is non-null
mask_multiple = df_prescreener[["U01_Email_18-49", "U01_Email_50+", "R24_Email"]].notna().sum(axis=1) > 1

# Print those rows to console if any
if mask_multiple.any():
    print("⚠️ Rows with more than one email:")
    print(df_prescreener.loc[mask_multiple, ["U01_Email_18-49", "U01_Email_50+", "R24_Email"]])

df_prescreener.loc[df_prescreener["type"] == "7", "ID"] += "_UMIP"
df_prescreener.loc[df_prescreener["type"] == "9", "ID"] += "_DCC"

df_prescreener.to_csv(os.path.join(SUBFILE_DIRECTORY, "Prescreener_Cleaned_2.csv"), index=False)


 ✦━━━━━━━━✧━━━━━━━━✦❘༻ CLEANING PRESCREENER ༺❘✦━━━━━━━━✧━━━━━━━━✦ 

ⓘ Dropping unused columns & rows...
DF Length:  13318
DF Length after Bots:  9333

 ✔ DONE! Check for Prescreener_cleaned.csv
DF Length after Email Unification:  9333


In [4]:

# (2) R24 & U01 LOGS ------------------------------------------------------------------------------------

print("\n ✦━━━━━━━━✧━━━━━━━━✦❘༻ LOADING LOGS ༺❘✦━━━━━━━━✧━━━━━━━━✦ \n")

# Load the CSV files without interpreting "NULL" as NaN
df_u01_logs = pd.read_csv(U01_LOGS_PATH, na_values=[], keep_default_na=False)
df_r24_logs_new = pd.read_csv(R24_LOGS_PATH, na_values=[], keep_default_na=False)
df_r24_logs_old = pd.read_csv(OLD_R24_LOGS_PATH, na_values=[], keep_default_na=False)

print("U01 LENGTH: ", len(df_u01_logs))
print("NEW R24 Length: ", len(df_r24_logs_new))
print("OLD R24 Length: ", len(df_r24_logs_old))

# Merge the R24 logs
df_r24_logs = pd.concat([df_r24_logs_old, df_r24_logs_new], ignore_index=True)

# Save the merged R24 logs
df_r24_logs.to_csv(os.path.join(SUBFILE_DIRECTORY, 'Merged_R24_Logs.csv'), index=False)

print("ⓘ Loaded logs")


 ✦━━━━━━━━✧━━━━━━━━✦❘༻ LOADING LOGS ༺❘✦━━━━━━━━✧━━━━━━━━✦ 

U01 LENGTH:  3873
NEW R24 Length:  1451
OLD R24 Length:  61
ⓘ Loaded logs


In [5]:
# (3) U01 POST ------------------------------------------------------------------------------------

print("\n ✦━━━━━━━━✧━━━━━━━━✦❘༻ CLEANING U01 POST ༺❘✦━━━━━━━━✧━━━━━━━━✦ \n")

df_u01_post = pd.read_csv(U01_POST_PATH, skiprows=[1, 2])

print("ⓘ Dropping unused columns & rows...")

# Drop the specified columns
columns_to_drop = [
    'RecipientLastName', 'RecipientFirstName', 'RecipientEmail',
    'ExternalReference',
    'DistributionChannel'
]

df_u01_post = df_u01_post.drop(columns=columns_to_drop)
print("U01 POST Length: ", len(df_u01_post))
df_u01_post = df_u01_post[
    (
        (df_u01_post["Q_RecaptchaScore"] >= 0.5) &
        (df_u01_post["Q_RelevantIDDuplicateScore"] < 75) &
        (df_u01_post["Q_RelevantIDFraudScore"] < 30)
    )
    | df_u01_post["ID"].isin(MUST_KEEP_IDS)
]

print("U01 POST Length after Bots: ", len(df_u01_post))

# (Optional) Save the result
df_u01_post.to_csv(os.path.join(SUBFILE_DIRECTORY, "U01_Post_cleaned.csv"), index=False)

print("\n ✔ DONE! Check for U01_Post_cleaned.csv")



 ✦━━━━━━━━✧━━━━━━━━✦❘༻ CLEANING U01 POST ༺❘✦━━━━━━━━✧━━━━━━━━✦ 

ⓘ Dropping unused columns & rows...
U01 POST Length:  2631
U01 POST Length after Bots:  1497

 ✔ DONE! Check for U01_Post_cleaned.csv


In [6]:
# (4) RENUMERATION ------------------------------------------------------------------------------------

print("\n ✦━━━━━━━━✧━━━━━━━━✦❘༻ R24 RENUMERATION ༺❘✦━━━━━━━━✧━━━━━━━━✦ \n")

df_r24_post = pd.read_csv(R24_POST_PATH, skiprows=[1, 2])
print("R24 POST Length: ", len(df_r24_post))

print("ⓘ Dropping unused columns & rows...")

# Drop the specified columns
columns_to_drop = [
    'RecipientLastName', 'RecipientFirstName', 'RecipientEmail',
    'ExternalReference',
    'DistributionChannel'
]

df_r24_post = df_r24_post.drop(columns=columns_to_drop)

df_r24_post = df_r24_post[
    (
        (df_r24_post["Q_RecaptchaScore"] >= 0.5) &
        (df_r24_post["Q_RelevantIDDuplicateScore"] < 75) &
        (df_r24_post["Q_RelevantIDFraudScore"] < 30)
    )
    | df_r24_post["ID"].isin(MUST_KEEP_IDS)
]

print("R24 POST Length AFTER BOTS: ", len(df_r24_post))

# (Optional) Save the result
df_r24_post.to_csv(os.path.join(SUBFILE_DIRECTORY, "R24_Post_cleaned.csv"), index=False)

print("\n ✔ DONE! Check for R24_Post_cleaned.csv")



 ✦━━━━━━━━✧━━━━━━━━✦❘༻ R24 RENUMERATION ༺❘✦━━━━━━━━✧━━━━━━━━✦ 

R24 POST Length:  1077
ⓘ Dropping unused columns & rows...
R24 POST Length AFTER BOTS:  933

 ✔ DONE! Check for R24_Post_cleaned.csv


In [7]:
# (5) RENUMERATION ------------------------------------------------------------------------------------

print("\n ✦━━━━━━━━✧━━━━━━━━✦❘༻ CLEANING RENUMERATION ༺❘✦━━━━━━━━✧━━━━━━━━✦ \n")

df_renumeration = pd.read_csv(RENUMERATION_PATH, skiprows=[1, 2])

print("ⓘ Dropping unused columns & rows...")

# Drop the specified columns
columns_to_drop = [
    'RecipientLastName', 'RecipientFirstName', 'RecipientEmail',
    'ExternalReference',
    'DistributionChannel'
]

df_renumeration = df_renumeration.drop(columns=columns_to_drop)
print("RENUMERATION Length: ", len(df_renumeration))


print("# of Participants per Score")
print("Q_RecaptchaScore:", (df_renumeration["Q_RecaptchaScore"] >= 0.5).sum())
print("Q_RelevantIDDuplicateScore:", (df_renumeration["Q_RelevantIDDuplicateScore"] < 75).sum())
print("Q_RelevantIDFraudScore:", (df_renumeration["Q_RelevantIDFraudScore"] < 30).sum())

print( " NO LONGER FILTERING OUT BY BOTS IN REMUNERATION " )

df_renumeration_old = df_renumeration[
    (
        (df_renumeration["Q_RecaptchaScore"] >= 0.5) &
        (df_renumeration["Q_RelevantIDDuplicateScore"] < 75) &
        (df_renumeration["Q_RelevantIDFraudScore"] < 30)
    )
    | df_renumeration["ID"].isin(MUST_KEEP_IDS)
]


print("RENUMERATION Length after bots: ", len(df_renumeration))

# (Optional) Save the result
df_renumeration.to_csv(os.path.join(SUBFILE_DIRECTORY, "Renumeration_cleaned.csv"), index=False)

print("\n ✔ DONE! Check for Renumeration_cleaned.csv")

# Define your mappings for Mayo/UM vs UF
mayo_um_mapping = {
    "Name_First": "Mayo/UM_Name_1",
    "Name_Last": "Mayo/UM_Name_4",
    "DOB_Month": "Mayo/UM_DOB#1_1",
    "DOB_Day": "Mayo/UM_DOB#2_1",
    "DOB_Year": "Mayo/UM_DOB#3_1",
    "Address_Street": "Mayo/UM_Address_4",
    "Address_City": "Mayo/UM_Address_5",
    "Address_State": "Mayo/UM_Address_6",
    "Address_Zip": "Mayo/UM_Address_7",
    "Email_Address": "Mayo/UM_Email_4",
    "Phone_Number": "Mayo/UM_Phone_4",
}

uf_mapping = {
    "Name_First": "UF_Name_1",
    "Name_Last": "UF_Name_4",
    "DOB_Month": "UF_DOB#1_1",
    "DOB_Day": "UF_DOB#2_1",
    "DOB_Year": "UF_DOB#3_1",
    "Address_Street": "UF_Address_4",
    "Address_City": "UF_Address_5",
    "Address_State": "UF_Address_6",
    "Address_Zip": "UF_Address_7",
    "Email_Address": "UF_Email_4",
    "Phone_Number": "UF_Phone_4",
}

# Conditions
for new_col in mayo_um_mapping.keys():
    mayo_col = mayo_um_mapping[new_col]
    uf_col = uf_mapping[new_col]

    # Start with NaNs
    df_renumeration[new_col] = np.nan

    # If the Mayo/UM column exists, fill it first
    if mayo_col in df_renumeration.columns:
        df_renumeration[new_col] = df_renumeration[mayo_col]

    # If the UF column exists, fill where Mayo/UM is missing
    if uf_col in df_renumeration.columns:
        df_renumeration[new_col] = df_renumeration[new_col].fillna(df_renumeration[uf_col])


print("RENUMERATION Length: ", len(df_renumeration))

df_renumeration.to_csv(os.path.join(SUBFILE_DIRECTORY, "Renumeration_cleaned_2.csv"), index=False)

print(df_renumeration.columns)



 ✦━━━━━━━━✧━━━━━━━━✦❘༻ CLEANING RENUMERATION ༺❘✦━━━━━━━━✧━━━━━━━━✦ 

ⓘ Dropping unused columns & rows...
RENUMERATION Length:  3553
# of Participants per Score
Q_RecaptchaScore: 2398
Q_RelevantIDDuplicateScore: 2595
Q_RelevantIDFraudScore: 2579
 NO LONGER FILTERING OUT BY BOTS IN REMUNERATION 
RENUMERATION Length after bots:  3553

 ✔ DONE! Check for Renumeration_cleaned.csv
RENUMERATION Length:  3553
Index(['StartDate', 'EndDate', 'Status', 'IPAddress', 'Progress',
       'Duration (in seconds)', 'Finished', 'RecordedDate', 'ResponseId',
       'LocationLatitude', 'LocationLongitude', 'UserLanguage',
       'Q_RecaptchaScore', 'Q_RelevantIDDuplicate',
       'Q_RelevantIDDuplicateScore', 'Q_RelevantIDFraudScore',
       'Q_RelevantIDLastStartDate', 'Q_DuplicateRespondent',
       'Q_BallotBoxStuffing', 'Q18', 'UMIP_Name_1', 'UMIP_Name_4',
       'UMIP_Email_4', 'UMIP_Phone_4', 'Mayo/UM_Name_1', 'Mayo/UM_Name_3',
       'Mayo/UM_Name_4', 'Mayo/UM_DOB#1_1', 'Mayo/UM_DOB#2_1',
       'M

In [8]:
def count_duplicates(series):
    series = series.dropna().astype(str).str.strip()
    series = series[series != ""]
    total = len(series)
    unique = series.nunique()
    duplicates = total - unique
    return total, unique, duplicates

def get_duplicate_values(df, col_name, id_col="ResponseId"):
    """
    Returns rows where `col_name` has duplicate values,
    excluding blanks and NaNs, along with their ResponseIds.
    """
    # Make a cleaned copy but keep index alignment
    clean_series = df[col_name].astype(str).str.strip()

    # Turn empty strings and "nan" back into real NaN
    clean_series = clean_series.replace(["", "nan", "NaN", "None"], np.nan)

    # Identify true duplicates (ignore NaNs)
    dupes_mask = clean_series.duplicated(keep=False) & clean_series.notna()

    # Subset with aligned mask
    dupes_df = df.loc[dupes_mask, [id_col, col_name]].copy()

    # Sort for readability
    dupes_df = dupes_df.sort_values(by=col_name, na_position="last")

    return dupes_df


print("\n ✦━━━━━━━━✧━━━━━━━━✦❘༻ CREATING COMPLETE DATASETS (PRE-LOGS-POST) ༺❘✦━━━━━━━━✧━━━━━━━━✦ \n")

print("Lengths so far: PRE, LOGS, U01 POST, R24, RENUM ALL")
print(len(df_prescreener))
print(len(df_u01_logs))
print(len(df_u01_post))
print(len(df_r24_post))
print(len(df_renumeration))

# Step 1: Merge df_logs and df_u01_post
df_pre_u01_logs = df_prescreener.merge(df_u01_logs, on='ID', how='inner')
df_pre_r24_logs = df_prescreener.merge(df_r24_logs, on='ID', how='inner')

print("Merged U01 Logs with Prescreener", len(df_pre_u01_logs))
print("Merged R24 Logs with Prescreener", len(df_pre_r24_logs))

print("ⓘ COMPLETE U01")

# Step 2: Merge the result with df_prescreener
merged_df_u01 = df_pre_u01_logs.merge(df_u01_post, on='ID', how='inner')
print("Merged U01 Logs, Pre, Post", len(merged_df_u01))

# 1. Create the IPAddress_match column
merged_df_u01['IPAddress_match'] = merged_df_u01['IPAddress_x'] == merged_df_u01['IPAddress_y']
print("Merged on IP: U01 Logs, Pre, Post", len(merged_df_u01))
print("IP Matches in x and y: ", (merged_df_u01['IPAddress_match'] == True).sum())

# 2. Convert the date columns to datetime
merged_df_u01['StartDate_x'] = pd.to_datetime(merged_df_u01['StartDate_x'])
merged_df_u01['EndDate_y'] = pd.to_datetime(merged_df_u01['EndDate_y'])

# Calculate the time difference in minutes and seconds as a string like "5m 32s"
time_diff = merged_df_u01['EndDate_y'] - merged_df_u01['StartDate_x']
merged_df_u01['TimeMinutes_Prescreener_to_PostSurvey'] = time_diff.dt.total_seconds().apply(
    lambda x: f"{int(x // 60)}m {int(x % 60)}s" if pd.notnull(x) else None
)

merged_df_u01.to_csv(os.path.join(SELECTED_OUTPUT_DIRECTORY, "U01_complete.csv"), index=False)

print("ⓘ COMPLETE R24")

# Step 2: Merge the result with df_prescreener
merged_df_r24 = df_pre_r24_logs.merge(df_r24_post, on='ID', how='inner')
print("Merged U01 Logs, Pre, Post", len(merged_df_r24))

# 1. Create the IPAddress_match column
merged_df_r24['IPAddress_match'] = merged_df_r24['IPAddress_x'] == merged_df_r24['IPAddress_y']
print("Merged on IP: R24 Logs, Pre, Post", len(merged_df_r24))

# 2. Convert the date columns to datetime
merged_df_r24['StartDate_x'] = pd.to_datetime(merged_df_r24['StartDate_x'])
merged_df_r24['EndDate_y'] = pd.to_datetime(merged_df_r24['EndDate_y'])

# Calculate the time difference in minutes and seconds as a string like "5m 32s"
time_diff = merged_df_r24['EndDate_y'] - merged_df_r24['StartDate_x']
merged_df_r24['TimeMinutes_Prescreener_to_PostSurvey'] = time_diff.dt.total_seconds().apply(
    lambda x: f"{int(x // 60)}m {int(x % 60)}s" if pd.notnull(x) else None
)

print("\n ✦━━━━━━━━✧━━━━━━━━✦❘༻ CREATING FINAL RENUMERATION FILES ༺❘✦━━━━━━━━✧━━━━━━━━✦ \n")
# 1️⃣ Extract base type
df_renumeration["type_renumeration"] = df_renumeration["type"].str.extract(r"^(U01|R24|u01|r24)")

# 2️⃣ Extract user ID from type if present
df_renumeration["user_id_from_type"] = df_renumeration["type"].str.extract(r"\?ID=(.+)$")

# 3️⃣ Combine with ID column to get final clean ID
df_renumeration["ID_renumeration"] = df_renumeration["ID"].replace("", pd.NA).combine_first(df_renumeration["user_id_from_type"])
# At this point we have :
# U01 - Containing a Merged Logs, Pre and Post
# R24 - Containing a Merged Logs, Pre and Post
# Filter, reset index, and make a safe copy
df_renumeration_u01 = df_renumeration[df_renumeration["type_renumeration"].str.lower() == "u01"].reset_index(drop=True).copy()

print ("Renumeration period: ", len(df_renumeration))
print("Renumeration u01: ", len(df_renumeration_u01))

# Filter, reset index, and make a safe copy
df_renumeration_r24 = df_renumeration[df_renumeration["type_renumeration"].str.lower() == "r24"].reset_index(drop=True).copy()

print("Renumeration r24: ", len(df_renumeration_r24))

# Step 1: Make sets for fast lookup
u01_complete_copy = merged_df_u01.copy()
r24_complete_copy = merged_df_r24.copy()

cols = ["Email_Address", "ID" ]

for col in cols:
    total, unique, duplicates = count_duplicates(u01_complete_copy[col])
    print(f"U01 COMPLETE {col}: total={total}, unique={unique}, duplicates={duplicates}")

for col in cols:
    dupes_df = get_duplicate_values(u01_complete_copy, col)
    if not dupes_df.empty:
        print(f"Duplicate {col} values (with ResponseId):")
        print(dupes_df)
    else:
        print(f"No duplicates found for {col}.")
    print("-" * 50)


for col in cols:
    total, unique, duplicates = count_duplicates(df_renumeration[col])
    print(f"RENUM {col}: total={total}, unique={unique}, duplicates={duplicates}")

for col in cols:
    dupes_df = get_duplicate_values(df_renumeration, col)
    if not dupes_df.empty:
        print(f"Duplicate {col} values (with ResponseId):")
        print(dupes_df)
    else:
        print(f"No duplicates found for {col}.")
    print("-" * 50)



# Clean and drop empty entries
u01_complete_email_column = (
    u01_complete_copy["Email_Address"]
    .dropna()                # remove NaN
    .astype(str)             # ensure string type
    .str.strip()             # remove whitespace
    .str.lower()             # make lowercase
)
u01_complete_email_column = u01_complete_email_column[u01_complete_email_column != ""]
u01_prescreener_emails = set(u01_complete_email_column)

u01_complete_ip_column = u01_complete_copy["IPAddress_x"].dropna()
u01_complete_ip_column = u01_complete_copy["IPAddress_x"].astype(str).str.strip()
u01_complete_ip_column = u01_complete_ip_column[u01_complete_ip_column != ""]
u01_ips = set(u01_complete_ip_column)

u01_complete_id_column = u01_complete_copy["ID"].dropna()
u01_complete_id_column = u01_complete_copy["ID"].astype(str).str.strip()
u01_complete_id_column = u01_complete_id_column[u01_complete_id_column != ""]
u01_ids = set(u01_complete_id_column)

r24_complete_id_column = r24_complete_copy["ID"].dropna()
r24_complete_id_column = r24_complete_copy["ID"].astype(str).str.strip()
r24_complete_id_column = r24_complete_id_column[r24_complete_id_column != ""]
r24_ids = set(r24_complete_id_column)

r24_complete_ip_column = r24_complete_copy["IPAddress_x"].dropna()
r24_complete_ip_column = r24_complete_copy["IPAddress_x"].astype(str).str.strip()
r24_complete_ip_column = r24_complete_ip_column[r24_complete_ip_column != ""]
r24_ips = set(r24_complete_ip_column)


u01_remuneration_id_column = df_renumeration_u01["ID_renumeration"].dropna()
u01_remuneration_id_column = df_renumeration_u01["ID_renumeration"].astype(str).str.strip()
u01_remuneration_id_column = u01_remuneration_id_column[u01_remuneration_id_column != ""]
u01_remuneration_ids = set(u01_remuneration_id_column)


r24_complete_email_column = (r24_complete_copy["Email_Address"].dropna().astype(str).str.strip().str.lower())
r24_complete_email_column = r24_complete_email_column[r24_complete_email_column != ""]
r24_prescreener_emails = set(r24_complete_email_column)

r24_remuneration_id_column = df_renumeration_r24["ID_renumeration"].dropna()
r24_remuneration_id_column = df_renumeration_r24["ID_renumeration"].astype(str).str.strip()
r24_remuneration_id_column = r24_remuneration_id_column[r24_remuneration_id_column != ""]
r24_remuneration_ids = set(r24_remuneration_id_column)




common_emails = set(df_renumeration_u01["Email_Address"]) & u01_prescreener_emails
common_ips = set(df_renumeration_u01["IPAddress"]) & u01_ips
common_ids = set(df_renumeration_u01["ID_renumeration"]) & u01_ids
r24_common_ids = set(df_renumeration_r24["ID_renumeration"]) & r24_ids
r24_common_emails = set(df_renumeration_r24["ID_renumeration"]) & r24_prescreener_emails

df_renumeration_u01["Email_Match"] = df_renumeration_u01["Email_Address"].astype(str).str.strip().str.lower().isin(u01_prescreener_emails) 
df_renumeration_u01["ID_Match"] = df_renumeration_u01["ID_renumeration"].astype(str).str.strip().isin(u01_ids)
df_renumeration_u01["IP_Match"] = df_renumeration_u01["IPAddress"].isin(u01_ips)

df_renumeration_r24["Email_Match"] = df_renumeration_r24["Email_Address"].astype(str).str.strip().str.lower().isin(r24_prescreener_emails) 
df_renumeration_r24["ID_Match"] = df_renumeration_r24["ID_renumeration"].astype(str).str.strip().isin(r24_ids) 
df_renumeration_r24["IP_Match"] = df_renumeration_r24["IPAddress"].isin(r24_ips)

MATCH_COLUMN = "Matched_Email_ID_or_IP_to_Pre"
df_renumeration_u01[MATCH_COLUMN] = df_renumeration_u01["Email_Match"] | df_renumeration_u01["ID_Match"] | df_renumeration_u01["IP_Match"] 
df_renumeration_r24[MATCH_COLUMN] = df_renumeration_r24["Email_Match"] | df_renumeration_r24["ID_Match"] | df_renumeration_r24["IP_Match"]



u01_remuneration_email_column = df_renumeration_u01["Email_Address"].dropna()
u01_remuneration_email_column = df_renumeration_u01["Email_Address"].astype(str).str.strip().str.lower()
u01_remuneration_email_column = u01_remuneration_email_column[u01_remuneration_email_column != ""]
u01_remuneration_emails = set(u01_remuneration_email_column)

u01_remuneration_ip_column = df_renumeration_u01["IPAddress"].dropna()
u01_remuneration_ip_column = df_renumeration_u01["IPAddress"].astype(str).str.strip()
u01_remuneration_ip_column = u01_remuneration_ip_column[u01_remuneration_ip_column != ""]
u01_remuneration_ips = set(u01_remuneration_ip_column)

u01_remuneration_id_column = df_renumeration_u01["ID_renumeration"].dropna()
u01_remuneration_id_column = df_renumeration_u01["ID_renumeration"].astype(str).str.strip()
u01_remuneration_id_column = u01_remuneration_id_column[u01_remuneration_id_column != ""]
u01_remuneration_ids = set(u01_remuneration_id_column)


r24_remuneration_email_column = df_renumeration_r24["Email_Address"].dropna()
r24_remuneration_email_column = df_renumeration_r24["Email_Address"].astype(str).str.strip().str.lower()
r24_remuneration_email_column = r24_remuneration_email_column[r24_remuneration_email_column != ""]
r24_remuneration_emails = set(r24_remuneration_email_column)

r24_remuneration_ip_column = df_renumeration_r24["IPAddress"].dropna()
r24_remuneration_ip_column = df_renumeration_r24["IPAddress"].astype(str).str.strip()
r24_remuneration_ip_column = r24_remuneration_ip_column[r24_remuneration_ip_column != ""]
r24_remuneration_ips = set(r24_remuneration_ip_column)

r24_remuneration_id_column = df_renumeration_r24["ID_renumeration"].dropna()
r24_remuneration_id_column = df_renumeration_r24["ID_renumeration"].astype(str).str.strip()
r24_remuneration_id_column = r24_remuneration_id_column[r24_remuneration_id_column != ""]
r24_remuneration_ids = set(r24_remuneration_id_column)



merged_df_u01["Email_Match"] = merged_df_u01["Email_Address"].astype(str).str.strip().str.lower().isin(u01_remuneration_emails) 
merged_df_u01["ID_Match"] = merged_df_u01["ID"].astype(str).str.strip().isin(u01_remuneration_ids)
merged_df_u01["IP_Match"] = merged_df_u01["IPAddress_x"].isin(u01_remuneration_ips)

merged_df_r24["Email_Match"] = merged_df_r24["Email_Address"].astype(str).str.strip().str.lower().isin(r24_remuneration_emails) 
merged_df_r24["ID_Match"] = merged_df_r24["ID"].astype(str).str.strip().isin(r24_remuneration_ids) 
merged_df_r24["IP_Match"] = merged_df_r24["IPAddress_x"].isin(r24_remuneration_ips)

MATCH_COLUMN_COMPLETE = "Matched_Email_ID_or_IP_to_Remun"
merged_df_u01[MATCH_COLUMN_COMPLETE] = merged_df_u01["Email_Match"] | merged_df_u01["ID_Match"] | merged_df_u01["IP_Match"] 
merged_df_r24[MATCH_COLUMN_COMPLETE] = merged_df_r24["Email_Match"] | merged_df_r24["ID_Match"] | merged_df_r24["IP_Match"]


print("Email Matches:", (df_renumeration_u01["Email_Match"] == True).sum())
print("ID Matches:", (df_renumeration_u01["ID_Match"] == True).sum())
print("IP Matches:", (df_renumeration_u01["IP_Match"] == True).sum())

print("R24 Email Matches: ", (df_renumeration_r24["Email_Match"] == True).sum())
print("R24 ID Matches: ", (df_renumeration_r24["ID_Match"] == True).sum())
print("R24 IP Matches:", (df_renumeration_u01["IP_Match"] == True).sum())

print("## Email Matches:", (merged_df_u01["Email_Match"] == True).sum())
print("ID Matches:", (merged_df_u01["ID_Match"] == True).sum())
print("IP Matches:", (merged_df_u01["IP_Match"] == True).sum())

print("R24 Email Matches: ", (merged_df_r24["Email_Match"] == True).sum())
print("R24 ID Matches: ", (merged_df_r24["ID_Match"] == True).sum())
print("R24 IP Matches:", (merged_df_r24["IP_Match"] == True).sum())




print("Unique Remuneration IDs for Remuneration: ", len(set(df_renumeration["ID_renumeration"])))
print("Unique Complete IDs for U01: ", len(u01_ids))
print("Unique Complete IDs for R24: ", len(r24_ids))
print("Unique Remuneration IDs for U01: ", len(u01_remuneration_ids))
print("Unique Remuneration IDs for R24: ", len(r24_remuneration_ids))
print("Common IDs in Complete and Remuneration for U01: ", len(common_ids))
print("Common IDs in Complete and Remuneration for R24: ", len (r24_common_ids))
not_in_common = u01_remuneration_ids - common_ids
r24_not_in_common = r24_remuneration_ids - r24_common_ids
print("Not in U01 common: ", not_in_common)
print("Not in R24 Common: " , r24_not_in_common)


u01_intermediate_renumeration = df_renumeration_u01.copy()
u01_intermediate_renumeration.to_csv(os.path.join(SUBFILE_DIRECTORY, "U01_Intermediary_Renum.csv"), index=False)

df_renumeration_u01 = df_renumeration_u01[[MATCH_COLUMN] + [c for c in df_renumeration_u01.columns if c != MATCH_COLUMN]]
df_renumeration_r24 = df_renumeration_r24[[MATCH_COLUMN] + [c for c in df_renumeration_r24.columns if c != MATCH_COLUMN]]
print("Either U01 Matches: ", (df_renumeration_u01[MATCH_COLUMN] == True).sum())
print("Either R24 Matches: ", (df_renumeration_r24[MATCH_COLUMN] == True).sum())

merged_df_u01 = merged_df_u01[[MATCH_COLUMN_COMPLETE] + [c for c in merged_df_u01.columns if c != MATCH_COLUMN_COMPLETE]]
merged_df_r24 = merged_df_r24[[MATCH_COLUMN_COMPLETE] + [c for c in merged_df_r24.columns if c != MATCH_COLUMN_COMPLETE]]
print("## Either U01 Matches: ", (merged_df_u01[MATCH_COLUMN_COMPLETE] == True).sum())
print("## Either R24 Matches: ", (merged_df_r24[MATCH_COLUMN_COMPLETE] == True).sum())


print("Regular renumeration: ", len(u01_intermediate_renumeration))
print("Final renumeration u01: ", len(df_renumeration_u01))
print("Complete U01: ", len(merged_df_u01))
print("Unique complete U01 U01: ", len(set(merged_df_u01["ID"])))

print("Final renumeration R24: ", len(df_renumeration_r24))
print("Complete R24: ", len(merged_df_r24))
print("Unique complete R24 R24: ", len(set(merged_df_r24["ID"])))




 ✦━━━━━━━━✧━━━━━━━━✦❘༻ CREATING COMPLETE DATASETS (PRE-LOGS-POST) ༺❘✦━━━━━━━━✧━━━━━━━━✦ 

Lengths so far: PRE, LOGS, U01 POST, R24, RENUM ALL
9333
3873
1497
933
3553
Merged U01 Logs with Prescreener 2947
Merged R24 Logs with Prescreener 1446
ⓘ COMPLETE U01
Merged U01 Logs, Pre, Post 1567
Merged on IP: U01 Logs, Pre, Post 1567
IP Matches in x and y:  1109
ⓘ COMPLETE R24
Merged U01 Logs, Pre, Post 1202
Merged on IP: R24 Logs, Pre, Post 1202

 ✦━━━━━━━━✧━━━━━━━━✦❘༻ CREATING FINAL RENUMERATION FILES ༺❘✦━━━━━━━━✧━━━━━━━━✦ 

Renumeration period:  3553
Renumeration u01:  2589
Renumeration r24:  959
U01 COMPLETE Email_Address: total=1546, unique=1150, duplicates=396
U01 COMPLETE ID: total=1567, unique=1190, duplicates=377
Duplicate Email_Address values (with ResponseId):
             ResponseId                 Email_Address
1145  R_1hg1qeVv2V46mtj       14daviswilson@gmail.com
1144  R_1hg1qeVv2V46mtj       14daviswilson@gmail.com
1146  R_1hg1qeVv2V46mtj       14daviswilson@gmail.com
1147  R_1

In [9]:
df_renumeration_u01 = df_renumeration_u01[
    df_renumeration_u01["Email_Match"] 
    | df_renumeration_u01["ID_Match"] 
    | df_renumeration_u01["IP_Match"]
].copy()

df_u01_temp = merged_df_u01[merged_df_u01[MATCH_COLUMN_COMPLETE] == False].copy()

# Fill in your desired values
df_u01_temp[MATCH_COLUMN] = False
df_u01_temp["StartDate"] = "N/A"
df_u01_temp["EndDate"] = "N/A"
df_u01_temp["ID_renumeration"] = df_u01_temp["ID"]

# ✅ Keep only those 4 columns
df_u01_temp = df_u01_temp[[MATCH_COLUMN, "StartDate", "EndDate", "ID_renumeration", "ID_Match", "Email_Match", "IP_Match"]]

# Append to main dataframe
df_renumeration_u01 = pd.concat([df_renumeration_u01, df_u01_temp], ignore_index=True)

# --- Step 1: normalize keys ---
df_renumeration_u01["ID_renumeration"] = df_renumeration_u01["ID_renumeration"].astype(str).str.strip()
df_renumeration_u01["Email_Address"] = df_renumeration_u01["Email_Address"].astype(str).str.strip().str.lower()
df_renumeration_u01["IPAddress"] = df_renumeration_u01["IPAddress"].astype(str).str.strip()

merged_df_u01["ID"] = merged_df_u01["ID"].astype(str).str.strip()
merged_df_u01["Email_Address"] = merged_df_u01["Email_Address"].astype(str).str.strip().str.lower()
merged_df_u01["IPAddress_x"] = merged_df_u01["IPAddress_x"].astype(str).str.strip()
df_renumeration_u01["Prescreener_Email"] = None

before = df_renumeration_u01["Prescreener_Email"].notna().sum()
id_to_email = (
    merged_df_u01.drop_duplicates(subset="ID")
    .set_index("ID")["Email_Address"]
)
df_renumeration_u01["Prescreener_Email"] = df_renumeration_u01["ID_renumeration"].map(id_to_email)
after = df_renumeration_u01["Prescreener_Email"].notna().sum()
print(f"✅ ID match filled {after - before} new Prescreener_Email values.")


before = after  # current count before this step
merged_df_u01["tempEmail"] = merged_df_u01["Email_Address"]
email_to_email = (merged_df_u01.drop_duplicates(subset="tempEmail")
                  .set_index("tempEmail")["Email_Address"])
mask = df_renumeration_u01["Prescreener_Email"].isna() | (df_renumeration_u01["Prescreener_Email"] == "")
df_renumeration_u01.loc[mask, "Prescreener_Email"] = (
    df_renumeration_u01.loc[mask, "Email_Address"].map(email_to_email)
)
after = df_renumeration_u01["Prescreener_Email"].notna().sum()
print(f"✅ Email match filled {after - before} new Prescreener_Email values.")


before = after
ip_to_email = (
    merged_df_u01.drop_duplicates(subset="IPAddress_x")
    .set_index("IPAddress_x")["Email_Address"]
)
mask = df_renumeration_u01["Prescreener_Email"].isna() | (df_renumeration_u01["Prescreener_Email"] == "")
df_renumeration_u01.loc[mask, "Prescreener_Email"] = (
    df_renumeration_u01.loc[mask, "IPAddress"].map(ip_to_email)
)
after = df_renumeration_u01["Prescreener_Email"].notna().sum()
print(f"✅ IP match filled {after - before} new Prescreener_Email values.")


df_renumeration_u01 = df_renumeration_u01.replace("nan", "")

df_renumeration_r24 = df_renumeration_r24[
    df_renumeration_r24["Email_Match"] 
    | df_renumeration_r24["ID_Match"] 
    | df_renumeration_r24["IP_Match"]
].copy()

df_r24_temp = merged_df_r24[merged_df_r24[MATCH_COLUMN_COMPLETE] == False].copy()

# Fill in your desired values
df_r24_temp[MATCH_COLUMN] = False
df_r24_temp["StartDate"] = "N/A"
df_r24_temp["EndDate"] = "N/A"
df_r24_temp["ID_renumeration"] = df_r24_temp["ID"]

# ✅ Keep only those 4 columns
df_r24_temp = df_r24_temp[[MATCH_COLUMN, "StartDate", "EndDate", "ID_renumeration", "ID_Match", "Email_Match", "IP_Match"]]

# Append to main dataframe
df_renumeration_r24 = pd.concat([df_renumeration_r24, df_r24_temp], ignore_index=True)

# --- Step 1: normalize keys ---
df_renumeration_r24["ID_renumeration"] = df_renumeration_r24["ID_renumeration"].astype(str).str.strip()
df_renumeration_r24["Email_Address"] = df_renumeration_r24["Email_Address"].astype(str).str.strip().str.lower()
df_renumeration_r24["IPAddress"] = df_renumeration_r24["IPAddress"].astype(str).str.strip()

merged_df_r24["ID"] = merged_df_r24["ID"].astype(str).str.strip()
merged_df_r24["Email_Address"] = merged_df_r24["Email_Address"].astype(str).str.strip().str.lower()
merged_df_r24["IPAddress_x"] = merged_df_r24["IPAddress_x"].astype(str).str.strip()
df_renumeration_r24["Prescreener_Email"] = None

before = df_renumeration_r24["Prescreener_Email"].notna().sum()
id_to_email = (
    merged_df_r24.drop_duplicates(subset="ID")
    .set_index("ID")["Email_Address"]
)
df_renumeration_r24["Prescreener_Email"] = df_renumeration_r24["ID_renumeration"].map(id_to_email)
after = df_renumeration_r24["Prescreener_Email"].notna().sum()
print(f"✅ ID match filled {after - before} new Prescreener_Email values.")


before = after  # current count before this step
merged_df_r24["tempEmail"] = merged_df_r24["Email_Address"]
email_to_email = (merged_df_r24.drop_duplicates(subset="tempEmail")
                  .set_index("tempEmail")["Email_Address"])
mask = df_renumeration_r24["Prescreener_Email"].isna() | (df_renumeration_r24["Prescreener_Email"] == "")
df_renumeration_r24.loc[mask, "Prescreener_Email"] = (
    df_renumeration_r24.loc[mask, "Email_Address"].map(email_to_email)
)
after = df_renumeration_r24["Prescreener_Email"].notna().sum()
print(f"✅ Email match filled {after - before} new Prescreener_Email values.")


before = after
ip_to_email = (
    merged_df_r24.drop_duplicates(subset="IPAddress_x")
    .set_index("IPAddress_x")["Email_Address"]
)
mask = df_renumeration_r24["Prescreener_Email"].isna() | (df_renumeration_r24["Prescreener_Email"] == "")
df_renumeration_r24.loc[mask, "Prescreener_Email"] = (
    df_renumeration_r24.loc[mask, "IPAddress"].map(ip_to_email)
)
after = df_renumeration_r24["Prescreener_Email"].notna().sum()
print(f"✅ IP match filled {after - before} new Prescreener_Email values.")


df_renumeration_r24 = df_renumeration_r24.replace("nan", "")

✅ ID match filled 1124 new Prescreener_Email values.
✅ Email match filled 179 new Prescreener_Email values.
✅ IP match filled 32 new Prescreener_Email values.
✅ ID match filled 940 new Prescreener_Email values.
✅ Email match filled 16 new Prescreener_Email values.
✅ IP match filled 2 new Prescreener_Email values.


In [10]:
merged_ids = merged_df_u01["ID"]
merged_set = set(merged_ids)
print(len(merged_set))

1190


In [11]:
# 1. Subset to duplicated IDs, ignoring empty strings
u01_remuneration_duplicates = (
    df_renumeration_u01[
        (df_renumeration_u01["ID_renumeration"].duplicated(keep=False)) &
        (df_renumeration_u01["ID_renumeration"].astype(str).str.strip() != "")
    ]
    .copy()
)

# 2. Columns required to be non-empty
required_cols = [
    "Name_First",
    "DOB_Month", "DOB_Day", "DOB_Year",
    "Address_Street", "Address_City", "Address_State", "Address_Zip",
    "Email_Address", "Phone_Number"
]

# 3. Flag rows that have all required values
u01_remuneration_duplicates["has_all_required"] = (
    u01_remuneration_duplicates[required_cols].notna().all(axis=1)
)

# 4. Preserve original order
u01_remuneration_duplicates["_orig_index"] = u01_remuneration_duplicates.index

# 5. Sort so first complete row per ID comes first
u01_remuneration_duplicates_sorted = u01_remuneration_duplicates.sort_values(
    by=["ID_renumeration", "has_all_required", "_orig_index"],
    ascending=[True, False, True]
)

# 6. Keep only ONE row per ID (first complete row if it exists)
u01_remuneration_deduped = u01_remuneration_duplicates_sorted.drop_duplicates(
    subset="ID_renumeration",
    keep="first"
)

u01_remuneration_deduped = (
    u01_remuneration_deduped
    .sort_values("_orig_index")
    .drop(columns=["has_all_required", "_orig_index"])
)

# 7. Remove all rows with duplicated IDs from original df, ignoring empty string IDs
df_no_duplicate_ids = df_renumeration_u01[
    ~df_renumeration_u01["ID_renumeration"].isin(
        u01_remuneration_duplicates["ID_renumeration"].unique()
    )
]

# 8. Combine deduped duplicates + non-duplicated rows
df_renumeration_nodups = pd.concat(
    [df_no_duplicate_ids, u01_remuneration_deduped],
    ignore_index=False
)

# 9. Restore original order
df_renumeration_nodups = df_renumeration_nodups.sort_index()

# Total number of duplicate rows (all rows in duplicated IDs)
total_duplicate_rows = u01_remuneration_duplicates.shape[0]

# Number of unique IDs that have duplicates
num_ids_with_duplicates = u01_remuneration_duplicates["ID_renumeration"].nunique()

# Length of original DataFrame
original_length = df_renumeration_u01.shape[0]

# Length of new deduplicated DataFrame
new_length = df_renumeration_nodups.shape[0]

print("===== Deduplication Summary =====")
print(f"Total number of duplicate rows (before deduping): {total_duplicate_rows}")
print(f"Number of unique IDs with duplicates: {num_ids_with_duplicates}")
print(f"Number of duplicates to be removed: {total_duplicate_rows - num_ids_with_duplicates}")
print(f"Original DataFrame length: {original_length}")
print(f"New DataFrame length (after deduping): {new_length}")
print(f"Difference of rows between original and new df: {original_length - new_length}")

===== Deduplication Summary =====
Total number of duplicate rows (before deduping): 196
Number of unique IDs with duplicates: 63
Number of duplicates to be removed: 133
Original DataFrame length: 1335
New DataFrame length (after deduping): 1202
Difference of rows between original and new df: 133


In [12]:
# 1. Subset to duplicated IDs, ignoring empty strings
r24_remuneration_duplicates = (
    df_renumeration_r24[
        (df_renumeration_r24["ID_renumeration"].duplicated(keep=False)) &
        (df_renumeration_r24["ID_renumeration"].astype(str).str.strip() != "")
    ]
    .copy()
)

# 2. Columns required to be non-empty
required_cols = [
    "Name_First",
    "DOB_Month", "DOB_Day", "DOB_Year",
    "Address_Street", "Address_City", "Address_State", "Address_Zip",
    "Email_Address", "Phone_Number"
]

# 3. Flag rows that have all required values
r24_remuneration_duplicates["has_all_required"] = (
    r24_remuneration_duplicates[required_cols].notna().all(axis=1)
)

# 4. Preserve original order
r24_remuneration_duplicates["_orig_index"] = r24_remuneration_duplicates.index

# 5. Sort so first complete row per ID comes first
r24_remuneration_duplicates_sorted = r24_remuneration_duplicates.sort_values(
    by=["ID_renumeration", "has_all_required", "_orig_index"],
    ascending=[True, False, True]
)

# 6. Keep only ONE row per ID (first complete row if it exists)
r24_remuneration_deduped = r24_remuneration_duplicates_sorted.drop_duplicates(
    subset="ID_renumeration",
    keep="first"
)

r24_remuneration_deduped = (
    r24_remuneration_deduped
    .sort_values("_orig_index")
    .drop(columns=["has_all_required", "_orig_index"])
)

# 7. Remove all rows with duplicated IDs from original df, ignoring empty string IDs
r24_no_duplicate_ids = df_renumeration_r24[
    ~df_renumeration_r24["ID_renumeration"].isin(
        r24_remuneration_duplicates["ID_renumeration"].unique()
    )
]

# 8. Combine deduped duplicates + non-duplicated rows
df_renumeration_r24_nodups = pd.concat(
    [r24_no_duplicate_ids, r24_remuneration_deduped],
    ignore_index=False
)

# 9. Restore original order
df_renumeration_r24_nodups = df_renumeration_r24_nodups.sort_index()

# Total number of duplicate rows (all rows in duplicated IDs)
total_duplicate_rows = r24_remuneration_duplicates.shape[0]

# Number of unique IDs that have duplicates
num_ids_with_duplicates = r24_remuneration_duplicates["ID_renumeration"].nunique()

# Length of original DataFrame
original_length = df_renumeration_r24.shape[0]

# Length of new deduplicated DataFrame
new_length = df_renumeration_r24_nodups.shape[0]

print("===== Deduplication Summary (R24) =====")
print(f"Total number of duplicate rows (before deduping): {total_duplicate_rows}")
print(f"Number of unique IDs with duplicates: {num_ids_with_duplicates}")
print(f"Number of duplicates to be removed: {total_duplicate_rows - num_ids_with_duplicates}")
print(f"Original DataFrame length: {original_length}")
print(f"New DataFrame length (after deduping): {new_length}")
print(f"Difference of rows between original and new df: {original_length - new_length}")

===== Deduplication Summary (R24) =====
Total number of duplicate rows (before deduping): 185
Number of unique IDs with duplicates: 24
Number of duplicates to be removed: 161
Original DataFrame length: 958
New DataFrame length (after deduping): 797
Difference of rows between original and new df: 161


In [13]:
df_renumeration_nodups = df_renumeration_nodups.drop(columns=["ResponseId"])

# 10. Save cleaned file
df_renumeration_nodups.to_csv(
    os.path.join(SELECTED_OUTPUT_DIRECTORY, "U01_Remuneration.csv"),
    index=False
)

df_renumeration_r24_nodups = df_renumeration_r24_nodups.drop(columns=["ResponseId"])

# 10. Save cleaned file
df_renumeration_r24_nodups.to_csv(
    os.path.join(SELECTED_OUTPUT_DIRECTORY, "R24_Remuneration.csv"),
    index=False
)

df_renumeration_u01 = merged_df_u01.drop(columns=["ResponseId"])
df_renumeration_u01 = merged_df_r24.drop(columns=["ResponseId"])

df_renumeration_u01.to_csv(os.path.join(SELECTED_OUTPUT_DIRECTORY, "U01_Remuneration_With_Duplicates.csv"), index=False)
df_renumeration_r24.to_csv(os.path.join(SELECTED_OUTPUT_DIRECTORY, "R24_Remuneration_With_Duplicates.csv"), index=False)

merged_df_u01 = merged_df_u01.drop(columns=["ResponseId"])
merged_df_r24 = merged_df_r24.drop(columns=["ResponseId"])

merged_df_u01.to_csv(os.path.join(SELECTED_OUTPUT_DIRECTORY, "U01_Complete.csv"), index=False)
merged_df_r24.to_csv(os.path.join(SELECTED_OUTPUT_DIRECTORY, "R24_Complete.csv"), index=False)